# KHILONA: Basic CNN Model (From Scratch)

This notebook implements a simple Convolutional Neural Network (CNN) for color classification of toy parts. 

### **Topics Covered:**
1. **Basics of Deep Learning**: Forward pass, loss, training loop.
2. **Machine Learning Theory**: Training vs Testing split (80/20 per class).
3. **CNN**: Basic Conv2D and MaxPooling architecture.

**NO PRETRAINED MODELS** - This model is built and trained entirely from scratch.

In [ ]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

np.random.seed(42)
random.seed(42)

In [ ]:
# Configuration
data_dir = "src/data"
img_size = 64
epochs = 15
batch_size = 32
classes = ["blue", "yellow", "purple"]
belts = ["A", "B", "C"]

In [ ]:
def load_and_split_data(root, classes, img_size, split=0.8):
    X_train, y_train = [], []
    X_test, y_test = [], []
    
    for idx, c in enumerate(classes):
        cls_dir = Path(root) / c
        files = list(cls_dir.glob("*.*"))
        random.shuffle(files)
        
        n_train = int(len(files) * split)
        train_files = files[:n_train]
        test_files = files[n_train:]
        
        print(f"Class {c}: {len(train_files)} training, {len(test_files)} testing")
        
        for p in train_files:
            try:
                img = Image.open(p).convert("RGB").resize((img_size, img_size))
                X_train.append(np.asarray(img, dtype=np.float32) / 255.0)
                y_train.append(idx)
            except: pass
            
        for p in test_files:
            try:
                img = Image.open(p).convert("RGB").resize((img_size, img_size))
                X_test.append(np.asarray(img, dtype=np.float32) / 255.0)
                y_test.append(idx)
            except: pass
            
    # Convert to NumPy arrays
    X_train, y_train = np.array(X_train), np.array(y_train)
    X_test, y_test = np.array(X_test), np.array(y_test)
    
    # One-hot encoding labels (Categorical)
    y_train = tf.keras.utils.to_categorical(y_train, num_classes=3)
    y_test = tf.keras.utils.to_categorical(y_test, num_classes=3)
    
    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_and_split_data(data_dir, classes, img_size)

In [ ]:
# Building the CNN Model - ABSOLUTELY NO PRETRAINED WEIGHTS
model = Sequential([
    # Topic 4: CNN Architecture
    Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    
    Flatten(),
    
    # Topic 3: MLP Part
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Model Training - NO Pre-trained models, NO Transfer Learning
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test, y_test)
)

In [ ]:
# Topic 2: Machine Learning Theory (Accuracy Score)
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"\n{'='*30}")
print(f"TRAINING ACCURACY: {train_acc:.4f}")
print(f"TESTING ACCURACY: {test_acc:.4f}")
print(f"{'='*30}")

In [ ]:
def predict_object(img_path):
    # Load and resize image
    img = Image.open(img_path).convert("RGB").resize((img_size, img_size))
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = np.expand_dims(x, axis=0)

    # Prediction
    prediction = model.predict(x, verbose=0)
    pred_idx = np.argmax(prediction)
    pred_class = classes[pred_idx]
    belt = belts[pred_idx]

    print(f"Predicted Color: {pred_class.upper()}")
    print(f"Assigned Conveyor Belt: {belt}")
    # print(f"Confidence: {prediction[0][pred_idx]:.4f}")
    
    plt.imshow(img)
    plt.axis('off')
    plt.show()

# To test, uncomment below:
# predict_object('path_to_test_image.jpg')

In [ ]:
import cv2

def run_live_camera():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Error: Could not open camera.")
        return

    print("Starting Live Detection... Press 'q' to quit.")
    
    while True:
        ret, frame = cap.read()
        if not ret: break

        # Basic OpenCV processing
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        resized_frame = cv2.resize(rgb_frame, (img_size, img_size))
        input_array = np.expand_dims(resized_frame / 255.0, axis=0)

        # Prediction
        prediction = model.predict(input_array, verbose=0)
        pred_idx = np.argmax(prediction)
        pred_class = classes[pred_idx]
        belt = belts[pred_idx]
        confidence = prediction[0][pred_idx]

        # Text Overlay
        label = f"{pred_class.upper()} | Belt: {belt}" # | {confidence:.2f}"
        cv2.putText(frame, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        cv2.imshow('KHILONA Live Detection', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# To run camera, uncomment below:
# run_live_camera()